# Multimodal ECG + Echo Preprocessing V3
Adapted for the 5-class cardiac fusion model. Runs offline on Windows.

**Changes from V2:**
- **ECG NaN handling:** NaN leads are replaced with zeros; fully-NaN records are skipped
- **Echo crop removed:** only 2% border trim instead of 30% — preserves full cardiac anatomy
- **5-class regrouping:** tail classes merged into "other" for both tasks
- **Validation:** built-in NaN audit and class distribution checks before saving

## 0. Install Dependencies

In [ ]:
# Run once — comment out after first install
# !pip install wfdb pydicom neurokit2 tqdm Pillow numpy pandas matplotlib scipy

## 1. Paths — Edit These

In [ ]:
import os

ECHO_DIR = r"C:\Users\anwme\Desktop\Datasets\Physionet\Physionet-ECHO"
ECG_DIR  = r"C:\Users\anwme\Desktop\Datasets\Physionet\Physionet-ECG"

CSV_PATH = r"C:\Users\anwme\Desktop\Preprocessing\final_dataset.csv"

# Where preprocessed .npz files will be saved
CACHE_DIR = r"C:\Users\anwme\Desktop\Datasets\cache_v3"

os.makedirs(CACHE_DIR, exist_ok=True)

print("=== ECHO directory ===")
echo_contents = os.listdir(ECHO_DIR)
print(f"Total items: {len(echo_contents)}")

print("\n=== ECG directory ===")
ecg_contents = os.listdir(ECG_DIR)
print(f"Total items: {len(ecg_contents)}")

## 2. Load CSV and Resolve Local Paths

Reads `final_dataset.csv`, resolves file paths to local directories, checks file availability, and displays class distributions before any processing.

In [ ]:
import pandas as pd
import os

df = pd.read_csv(CSV_PATH)
print(f"Rows in CSV: {len(df)}")

def last_3_parts(p):
    parts = p.replace("\\", "/").split("/")
    return os.path.join(*parts[-3:])

df['ecg_rel']      = df['ecg_path'].apply(last_3_parts)
df['echo_ED_rel']  = df['echo_path_ED'].apply(last_3_parts)
df['echo_Mid_rel'] = df['echo_path_Mid'].apply(last_3_parts)
df['echo_ES_rel']  = df['echo_path_ES'].apply(last_3_parts)

df['ecg_full']  = df['ecg_rel'].apply(lambda x: os.path.join(ECG_DIR, x))
df['echo_ED']   = df['echo_ED_rel'].apply(lambda x: os.path.join(ECHO_DIR, x))
df['echo_Mid']  = df['echo_Mid_rel'].apply(lambda x: os.path.join(ECHO_DIR, x))
df['echo_ES']   = df['echo_ES_rel'].apply(lambda x: os.path.join(ECHO_DIR, x))

def row_available(r):
    return (os.path.exists(r['ecg_full'] + '.hea') and
            os.path.exists(r['ecg_full'] + '.dat') and
            os.path.exists(r['echo_ED'])  and
            os.path.exists(r['echo_Mid']) and
            os.path.exists(r['echo_ES']))

print("\nScanning local files...")
df['available'] = df.apply(row_available, axis=1)
df_avail = df[df['available']].copy()

print(f"\nAvailable: {len(df_avail)} / {len(df)}")
print(f"\nSplit counts:\n{df_avail['split'].value_counts()}")
print(f"\nstructural_coarse distribution:\n{df_avail['structural_coarse'].value_counts()}")
print(f"\narrhythmia_coarse distribution:\n{df_avail['arrhythmia_coarse'].value_counts()}")

df_avail.to_csv(os.path.join(CACHE_DIR, 'df_avail.csv'), index=False)
print(f"\nSaved df_avail.csv")

## 3. Define 5-Class Mapping

Tail classes with fewer than ~200 samples are merged into "other" categories:
- **Structural:** valve_disease + pericardial_disease + congenital → `other_structural`
- **Arrhythmia:** tachycardia + bradycardia + other_arrhythmia → `other_arrhythmia`

This gives 5 classes per task with no class below ~170 samples.

In [ ]:
STRUCT_CLASSES = [
    'ischemic_heart_disease', 'heart_failure', 'normal',
    'cardiomyopathy', 'other_structural',
]
ARR_CLASSES = [
    'atrial_fibrillation', 'normal', 'ventricular_arrhythmia',
    'conduction_disorder', 'other_arrhythmia',
]

NUM_CLASSES_STRUCT = len(STRUCT_CLASSES)
NUM_CLASSES_ARR    = len(ARR_CLASSES)

STRUCT_MAP = {
    'ischemic_heart_disease': 0,
    'heart_failure':          1,
    'normal':                 2,
    'cardiomyopathy':         3,
    'valve_disease':          4,
    'pericardial_disease':    4,
    'congenital':             4,
}

ARR_MAP = {
    'atrial_fibrillation':    0,
    'normal':                 1,
    'ventricular_arrhythmia': 2,
    'conduction_disorder':    3,
    'tachycardia':            4,
    'bradycardia':            4,
    'other_arrhythmia':       4,
}

print(f"Structural ({NUM_CLASSES_STRUCT}): {STRUCT_CLASSES}")
print(f"Arrhythmia ({NUM_CLASSES_ARR}): {ARR_CLASSES}")

df_avail = pd.read_csv(os.path.join(CACHE_DIR, 'df_avail.csv'))
unmapped_s = set(df_avail['structural_coarse'].unique()) - set(STRUCT_MAP.keys())
unmapped_a = set(df_avail['arrhythmia_coarse'].unique()) - set(ARR_MAP.keys())
if unmapped_s:
    print(f"WARNING: unmapped structural labels: {unmapped_s}")
if unmapped_a:
    print(f"WARNING: unmapped arrhythmia labels: {unmapped_a}")
if not unmapped_s and not unmapped_a:
    print("All labels mapped successfully.")

## 4. Preprocessing Functions

**ECG preprocessing:**
- Load via `wfdb.rdrecord`
- Replace NaN values with 0 per-lead before normalisation
- Per-lead z-score normalisation, clipped to ±8
- Pad or truncate to exactly 5000 samples
- Skip records where ALL 12 leads are NaN/constant

**Echo preprocessing:**
- Load DICOM, handle YBR colour space conversion
- Minimal 2% border trim (just enough to remove DICOM frame borders)
- Convert to grayscale
- Resize to 224×224 with bilinear interpolation

In [ ]:
import wfdb
import pydicom
import numpy as np
from pydicom.pixels import convert_color_space
from PIL import Image
import warnings, logging

warnings.filterwarnings('ignore', category=UserWarning, module='pydicom')
logging.getLogger('pydicom').setLevel(logging.ERROR)

ECHO_SIZE = 224
ECG_LEN   = 5000
MAX_DCM_BYTES   = 15 * 1024 * 1024
MAX_PIXEL_BYTES = 200 * 1024 * 1024


def preprocess_ecg(path_no_ext):
    rec = wfdb.rdrecord(path_no_ext)
    sig = rec.p_signal.astype(np.float32).T  # (12, T)

    # Replace NaN with 0 per-lead
    for lead in range(sig.shape[0]):
        nan_mask = np.isnan(sig[lead])
        if nan_mask.all():
            sig[lead] = 0.0
        elif nan_mask.any():
            sig[lead, nan_mask] = 0.0

    # Pad or truncate to ECG_LEN
    T = sig.shape[1]
    if T < ECG_LEN:
        sig = np.pad(sig, ((0, 0), (0, ECG_LEN - T)), mode='constant')
    elif T > ECG_LEN:
        sig = sig[:, :ECG_LEN]

    # Per-lead z-score normalisation
    mean = sig.mean(axis=1, keepdims=True)
    std  = sig.std(axis=1, keepdims=True)
    valid = std.squeeze() > 1e-6
    sig[valid] = np.clip((sig[valid] - mean[valid]) / std[valid], -8, 8)
    sig[~valid] = 0.0

    return sig.astype(np.float32)


def preprocess_echo(dcm_path):
    fsize = os.path.getsize(dcm_path)
    if fsize > MAX_DCM_BYTES:
        raise ValueError(f"DICOM too large: {fsize/1e6:.1f} MB")

    dcm = pydicom.dcmread(dcm_path, stop_before_pixels=True)
    rows    = int(getattr(dcm, 'Rows', 0))
    cols    = int(getattr(dcm, 'Columns', 0))
    frames  = int(getattr(dcm, 'NumberOfFrames', 1))
    samples = int(getattr(dcm, 'SamplesPerPixel', 1))
    bits    = int(getattr(dcm, 'BitsAllocated', 8))
    photo   = dcm.PhotometricInterpretation
    estimated_bytes = rows * cols * frames * samples * (bits // 8)
    del dcm

    if estimated_bytes > MAX_PIXEL_BYTES:
        raise ValueError(f"Pixel array would be {estimated_bytes/1e6:.0f} MB")

    dcm = pydicom.dcmread(dcm_path)
    px  = dcm.pixel_array
    del dcm

    if photo == 'YBR_FULL_422':
        px = convert_color_space(px, 'YBR_FULL_422', 'RGB')

    if px.ndim == 4:
        px = px[px.shape[0] // 2].copy()

    # Minimal 2% border trim (just DICOM frame edges)
    H, W = px.shape[:2]
    t, b = int(H * 0.02), int(H * 0.98)
    l, r = int(W * 0.02), int(W * 0.98)
    px = px[t:b, l:r]

    if px.ndim == 3:
        gray = (0.299 * px[..., 0] + 0.587 * px[..., 1] + 0.114 * px[..., 2]).astype(np.uint8)
    else:
        gray = px.astype(np.uint8)
    del px

    gray = np.array(
        Image.fromarray(gray).resize((ECHO_SIZE, ECHO_SIZE), Image.BILINEAR),
        dtype=np.uint8,
    )
    return gray


# Quick sanity check
print("Testing preprocessing functions...")
row = df_avail.iloc[0]
ecg = preprocess_ecg(row['ecg_full'])
ed  = preprocess_echo(row['echo_ED'])
print(f"  ECG: shape={ecg.shape}, NaN={np.isnan(ecg).any()}, mean={ecg.mean():.4f}")
print(f"  Echo ED: shape={ed.shape}, range=[{ed.min()}, {ed.max()}]")
print("Preprocessing functions OK.")

## 5. Full Preprocessing Pipeline

Processes all available samples and saves `train.npz`, `val.npz`, `test.npz` + `classes.json`.

Checkpoints progress every ~10% so you can resume if interrupted.

**Important:** if re-running, delete old cache files first or use a new `CACHE_DIR`.

In [ ]:
import os, numpy as np, pandas as pd, wfdb, pydicom
import json, gc, shutil, warnings, logging
from pydicom.pixels import convert_color_space
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=UserWarning, module='pydicom')
logging.getLogger('pydicom').setLevel(logging.ERROR)

df = pd.read_csv(os.path.join(CACHE_DIR, 'df_avail.csv'))
print(f"Total rows: {len(df)}")


def process_one(i, row):
    try:
        ecg = preprocess_ecg(row['ecg_full'])
        if np.abs(ecg).max() < 1e-4:
            return None, None, None, None, "ECG all-zero after normalisation"

        ed  = preprocess_echo(row['echo_ED'])
        mid = preprocess_echo(row['echo_Mid'])
        es  = preprocess_echo(row['echo_ES'])
        echo = np.stack([ed, mid, es])

        lbl_s = STRUCT_MAP[row['structural_coarse']]
        lbl_a = ARR_MAP[row['arrhythmia_coarse']]
        return ecg, echo, lbl_s, lbl_a, None
    except Exception as e:
        return None, None, None, None, str(e)


for split_name in ['train', 'val', 'test']:
    final_path = os.path.join(CACHE_DIR, f'{split_name}.npz')
    if os.path.exists(final_path):
        print(f"\n{split_name}.npz already exists - skipping")
        continue

    subset    = df[df['split'] == split_name].reset_index(drop=True)
    N         = len(subset)
    split_dir = os.path.join(CACHE_DIR, f'{split_name}_samples')
    os.makedirs(split_dir, exist_ok=True)

    log_path = os.path.join(CACHE_DIR, f'{split_name}_done.json')
    done = {}
    if os.path.exists(log_path):
        with open(log_path) as f:
            done = json.load(f)
        done = {
            k: v for k, v in done.items()
            if v == 'failed' or os.path.exists(os.path.join(split_dir, f'{k}.npz'))
        }

    print(f"\n=== {split_name}: {N} samples, {len(done)} already done ===")
    todo = [i for i in range(N) if str(i) not in done]
    ckpt_every = max(1, N // 10)

    def save_log():
        with open(log_path, 'w') as f:
            json.dump(done, f)

    pbar = tqdm(todo, initial=len(done), total=N, desc=split_name)
    since_ckpt = 0
    for i in pbar:
        row = subset.iloc[i]
        ecg, echo, lbl_s, lbl_a, err = process_one(i, row)
        if ecg is not None:
            np.savez(os.path.join(split_dir, f'{i}.npz'),
                     ecg=ecg, echo=echo,
                     label_struct=lbl_s, label_arr=lbl_a)
            done[str(i)] = 'ok'
            del ecg, echo
        else:
            done[str(i)] = 'failed'
        since_ckpt += 1
        if since_ckpt >= ckpt_every:
            save_log()
            since_ckpt = 0
            ok     = sum(1 for v in done.values() if v == 'ok')
            failed = sum(1 for v in done.values() if v == 'failed')
            pbar.set_postfix(ok=ok, failed=failed)
            gc.collect()
    save_log()
    gc.collect()

    ok_indices = sorted([int(k) for k, v in done.items() if v == 'ok'])
    failed_count = sum(1 for v in done.values() if v == 'failed')
    if not ok_indices:
        print(f"  No valid samples for {split_name}")
        continue

    M = len(ok_indices)
    print(f"  Assembling {split_name}.npz ({M} samples, {failed_count} failed)...")
    ecgs     = np.zeros((M, 12, ECG_LEN),            dtype=np.float32)
    echoes   = np.zeros((M, 3, ECHO_SIZE, ECHO_SIZE), dtype=np.uint8)
    labels_s = np.zeros(M, dtype=np.int64)
    labels_a = np.zeros(M, dtype=np.int64)

    for j, i in enumerate(tqdm(ok_indices, desc='Assemble')):
        with np.load(os.path.join(split_dir, f'{i}.npz')) as d:
            ecgs[j]     = d['ecg']
            echoes[j]   = d['echo']
            labels_s[j] = int(d['label_struct'])
            labels_a[j] = int(d['label_arr'])

    # NaN audit
    nan_samples = np.isnan(ecgs).any(axis=(1, 2)).sum()
    if nan_samples > 0:
        print(f"  WARNING: {nan_samples} samples with NaN - removing them")
        clean = ~np.isnan(ecgs).any(axis=(1, 2))
        ecgs, echoes = ecgs[clean], echoes[clean]
        labels_s, labels_a = labels_s[clean], labels_a[clean]
        M = len(ecgs)
        print(f"  After cleanup: {M} samples")

    np.savez(final_path, ecg=ecgs, echo=echoes,
             label_struct_coarse=labels_s,
             label_arr_coarse=labels_a)

    print(f"  Done: {split_name}: {M} samples -> {final_path}")
    shutil.rmtree(split_dir)
    del ecgs, echoes, labels_s, labels_a
    gc.collect()

classes_path = os.path.join(CACHE_DIR, 'classes.json')
with open(classes_path, 'w') as f:
    json.dump({
        'struct_classes': STRUCT_CLASSES,
        'struct_map': STRUCT_MAP,
        'arr_classes': ARR_CLASSES,
        'arr_map': ARR_MAP,
        'num_classes_struct': NUM_CLASSES_STRUCT,
        'num_classes_arr': NUM_CLASSES_ARR,
    }, f, indent=2)

print(f"\nAll done. Output files:")
for fname in ['train.npz', 'val.npz', 'test.npz', 'classes.json']:
    p = os.path.join(CACHE_DIR, fname)
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / 1e6
        print(f"  {fname:20s} {size_mb:8.1f} MB")

## 6. Verify Output

Checks shapes, dtypes, NaN counts, and class distributions. Also shows sample ECG + Echo frames from different classes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json, os

with open(os.path.join(CACHE_DIR, 'classes.json')) as f:
    meta = json.load(f)

STRUCT_CLASSES = meta['struct_classes']
ARR_CLASSES    = meta['arr_classes']

for split in ['train', 'val', 'test']:
    print(f"\n{'='*60}")
    print(f"  {split.upper()}")
    print(f"{'='*60}")
    data = np.load(os.path.join(CACHE_DIR, f'{split}.npz'))
    ecg     = data['ecg']
    echo    = data['echo']
    label_s = data['label_struct_coarse']
    label_a = data['label_arr_coarse']

    print(f"  ECG   : {ecg.shape}  dtype={ecg.dtype}")
    print(f"  Echo  : {echo.shape}  dtype={echo.dtype}")
    print(f"  Labels: struct={label_s.shape}  arr={label_a.shape}")

    nan_count = np.isnan(ecg).any(axis=(1, 2)).sum()
    print(f"  ECG NaN samples: {nan_count} / {len(ecg)}  {'OK' if nan_count == 0 else 'WARNING'}")
    print(f"  ECG mean={ecg.mean():.4f}  std={ecg.std():.4f}")

    print(f"\n  Structural distribution:")
    for c, n in zip(*np.unique(label_s, return_counts=True)):
        print(f"    {STRUCT_CLASSES[c]:30s} {n:5d}")

    print(f"\n  Arrhythmia distribution:")
    for c, n in zip(*np.unique(label_a, return_counts=True)):
        print(f"    {ARR_CLASSES[c]:30s} {n:5d}")

# Visual sanity check
data = np.load(os.path.join(CACHE_DIR, 'train.npz'))
ecg, echo = data['ecg'], data['echo']
label_s, label_a = data['label_struct_coarse'], data['label_arr_coarse']

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
shown = set()
idx = 0
for i in range(len(ecg)):
    cls = int(label_s[i])
    if cls in shown:
        continue
    shown.add(cls)

    axes[idx, 0].plot(ecg[i, 1, :2500])
    axes[idx, 0].set_title(f"ECG Lead II\n{STRUCT_CLASSES[cls]}", fontsize=9)

    for j, (frame, name) in enumerate(zip(echo[i], ['ED', 'Mid', 'ES'])):
        axes[idx, j+1].imshow(frame, cmap='gray')
        axes[idx, j+1].set_title(name, fontsize=9)
        axes[idx, j+1].axis('off')

    idx += 1
    if idx >= 3:
        break

plt.suptitle('Sample ECG + Echo (3 structural classes)', fontsize=13)
plt.tight_layout()
plt.show()
print("\nPreprocessing verification complete.")